In [ ]:
import json

localpath = "C:/Users/nikla/Documents/HiwiStuttgart/munitquest-data-submission"

# app/input is where folder lives.

import subprocess

In [ ]:
# creating a validation json and then writing it into a file, because this way i can put comments here, which directly editing the .json file would not allow.

validationDict = { # JSON configuration file that reclassifies issues as warnings, errors or ignored.
    "ignore": [ 
        {
            "code": "SIDECAR_WITHOUT_DATAFILE",  # ignore missing edfs
        }, 
        {
            "subCode": "HEDVersion"
        },
    ],
    "warning": [],
    "error": [],
}


with open(localpath + "/bundle/scoring/config.json", "w") as f:
    json.dump(validationDict, f, indent=4)

In [ ]:
def run_bids_validator(
    path,
    ignored_codes=[],
    ignored_fields=[],
    ignored_files=[],
    print_errors=True,
    print_warnings=True,
):
    """
    API to the official BIDS validator.

    Args:
        path (str): Absolute of relative path to your BIDS dataset
        ignored_codes (list of str): Ignored error codes (e.g. ["SIDECAR_KEY_RECOMMENDED"])
        ignored_fileds (list of str): Errors corresponding to that field are ignored (e.g. ["DeviceSerialNumber"])
        ignored_files (list of str): Ignored errors in these files (e.g. ["/dataset_description.json"])
        print_errors (bool): Descides if errors should be printed
        print_warnings (bool): Descides if warnings should be printed

    Returns:
        errors (list): List of detected errors
        warnings (list): List of detected warnings
        valid (bool): True if there are no errors

    """

    # Run bids validator
    result = subprocess.run(
        [
            "bids-validator-deno",
            "--format",
            "json",
            "--config",
            localpath + "/bundle/scoring/config.json",  # specifies config file to use
            path,
        ],
        capture_output=True,
        text=True,
    )
    # Extract and filter all issues
    validation = json.loads(result.stdout)
    issues = validation["issues"]["issues"]
    issues = [f for f in issues if (not "code" in f or f["code"] not in ignored_codes)]
    issues = [
        f for f in issues if (not "subCode" in f or f["subCode"] not in ignored_fields)
    ]
    issues = [
        f for f in issues if (not "location" in f or f["location"] not in ignored_files)
    ]
    # Split issues in errors and warnings
    errors = [f for f in issues if f["severity"] == "error"]
    warnings = [f for f in issues if f["severity"] == "warning"]
    # Print issues
    if print_errors:
        print(f"Number of detected errors: {len(errors)}")
        print(json.dumps(errors, indent=4))
    if print_warnings:
        print(f"Number of detected warnings: {len(warnings)}")
        print(json.dumps(warnings, indent=4))
    # Check if the folder is BIDS valid
    valid = True if len(errors) == 0 else False

    return errors, warnings, valid


def run_bids_validator_raw(path):

    result = subprocess.run(
        [
            "bids-validator-deno",
            "--format",
            "json",
            "--config",
            localpath + "/bundle/scoring/config.json",  # specifies config file to use
            path,
        ],
        capture_output=True,
        text=True,
    )
    return result

In [ ]:
# result = run_bids_validator_raw(localpath + "/app/input")
# neat = json.dumps(json.loads(result.stdout), indent=2)
# print(neat)

In [ ]:
errors, warnings, valid = run_bids_validator(
    localpath + "/app/input",
    ignored_codes=[],
    ignored_fields=[],
    ignored_files=[],
    print_errors=True,
    print_warnings=True,
)



In [ ]:
# also write validation result into file, just in case?

In [ ]:
placeholder: dict[str, float] = {"valid": 1}  # 1 if passed, 0 if not.

with open(localpath + "/app/output/scores.json", "w") as f:
    json.dump(placeholder, f, indent=4)